# Through Time

**A machine looks at real artworks, from the first human drawings to Van Gogh, and makes its own piece from them.**

1. **Collect**: finds and downloads openly licensed works by itself (Wikimedia Commons cave paintings with reusable licences, The Met Open Access, Art Institute of Chicago). Every source and licence is saved in `credits.csv`.
2. **Look**: a vision model (CLIP) turns each work into a vector.
3. **Read**: how strongly each work matches five ways of showing a person (symbolic, lifelike, emotional, fragmented, dissolving into light), over time.
4. **Atlas**: all works on one map; near means "looks alike to the machine".
5. **Journey**: the machine's own piece. Its gaze travels through the works in time order and light flows through each work's own lines.

Run everything with **Runtime → Run all**. A GPU makes step 2 faster (Runtime → Change runtime type → T4 GPU) but is not required.

In [ ]:
!git clone -q https://github.com/ilaydayilmazz/swarm-gaze.git
%cd swarm-gaze
!pip -q install -r requirements.txt open_clip_torch umap-learn matplotlib

## 1. Collect
About 100 works. Change the numbers or add artists (only artists whose works are in the public domain).

In [ ]:
import through_time as tt
tt.collect(out="data/time", per_period=12, cave=12, artists=("Vincent van Gogh",), per_artist=15)
rows = tt.load_collection("data/time")
print(len(rows), "works, from", int(rows[0]["year"]), "to", int(rows[-1]["year"]))

In [ ]:
# What was collected, oldest first
from PIL import Image
import matplotlib.pyplot as plt
pick = rows[:: max(1, len(rows) // 24)][:24]
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
for ax, r in zip(axes.flat, pick):
    ax.imshow(Image.open(r["path"]).convert("RGB")); ax.set_title(f'{int(r["year"])}\n{r["period"]}', fontsize=8); ax.axis("off")
for ax in list(axes.flat)[len(pick):]: ax.axis("off")
plt.tight_layout(); plt.show()

## 2–3. Look and read

In [ ]:
vecs, scores = tt.look(rows)
from IPython.display import Image as Show
Show(tt.plot_readings(rows, scores))

## 4. Atlas

In [ ]:
Show(tt.atlas(rows, vecs), width=900)

## 5. Journey: the machine's own piece

In [ ]:
tt.journey(rows, out="out_time", seconds_per_work=1.6)
from IPython.display import Video
Video("out_time/journey.mp4", embed=True, width=900)

## Credits
Every work used, with its source and licence. Keep this file with anything you publish.

In [ ]:
import pandas as pd
pd.read_csv("data/time/credits.csv")